# Additional Experiment — Supplement Configs (Extraction + Probing A/B)

Mirrors `experiment.ipynb`, but for **vote buckets that already have generations + judge
votes** and were unused by the main 3-class experiment. Re-labeled directly from the
existing `*_full.csv` — **no knowledge test, no generation, no judging**.

New classes (`SUPPLEMENT_LABEL_RULES` in `utils/settings.py`):

| label | bucket (config / votes_correct) | meaning |
|---|---|---|
| `natural_deception` | A / 0 | passed KC, neutral prompt, all-wrong |
| `capable_failed` | B / 6 | failed KC, neutral prompt, all-right |
| `deception_rejection` | C / 6 | passed KC, deceptive prompt, all-right |

**Pipeline:** load judged data → build supplement dataset → extract activations (new
classes) → combine with the original 3 classes → PCA → **probing A + B**.

**Probing replaces the old direct-3way / cascaded / binary suite with two views:**
- **A — multi-class landscape (LR):** which categories are easy/hard to separate.
  Run on the neutral 2×2 (truth / natural_deception / capable_failed / honest_mistake — a
  clean *knowledge × answer-correctness* study with no deceptive-prompt confound) and on
  all classes.
- **B — targeted binaries (LR, AUROC):** one sharp question per pair (AUROC ≈ 0.5 ⇒
  internally indistinguishable). Defined by `SUPPLEMENT_BINARY_PAIRS`.

MLP, plots, and the old cascaded probes are intentionally deferred/dropped (see final note).

**Change any parameter in `utils/settings.py` — never here:** `MODEL_ID` / `RUN_SLUG`
(qwen2.5 → qwen3 → gemma-4 needs no notebook change), `SUPPLEMENT_LABEL_RULES`,
`SUPPLEMENT_BINARY_PAIRS`, `PCA_K`, `N_SPLITS`, `MAX_ITER`, `RANDOM_STATE`.

## Part 1: Setup & Load Model

In [1]:
import os
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters.
from utils.settings import *

# Utils (reused — this notebook has no bespoke extraction/probing logic)
from utils.analysis import build_probe_dataset, split_thinking_responses, run_pca_reduction
from utils.activation import run_extract_activations, drop_overlong_rows, LABEL_MAP
from utils.probe import probe_all_layers, probe_all_layers_binary

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Output directories (distinct additional-config filenames/folders live here)
for d in [
    SUPPLEMENT_PROBE_DATASET_PATH.parent, SUPPLEMENT_ACTIVATIONS_PATH.parent,
    SUPPLEMENT_MULTICLASS_LR_DIR, SUPPLEMENT_BINARY_LR_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Model:    {MODEL_ID}")
print(f"Run slug: {RUN_SLUG or '(none)'}")
print(f"Supplement rules: {SUPPLEMENT_LABEL_RULES}")
print("Reading judged data from:")
print(f"  {TRUTHFULQA_FULL_PATH}")
print(f"  {MMLU_FULL_PATH}")
print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda" and torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Model:    Qwen/Qwen3-4B
Run slug: non_thinking_mode/original_deception_prompt
Supplement rules: [('A', 0, 'natural_deception'), ('B', 6, 'capable_failed'), ('C', 6, 'deception_rejection')]
Reading judged data from:
  data/dataset/qwen3-4b/non_thinking_mode/original_deception_prompt/judge/truthfulQA_full.csv
  data/dataset/qwen3-4b/non_thinking_mode/original_deception_prompt/judge/mmlu_full.csv

Device: cuda
GPU:  NVIDIA L40S
VRAM: 47.7 GB


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_READ_TOKEN)

# GPU memory cap derived from the actual card (was hard-coded 22GiB, tuned for a 24GB 4090).
# Leave ~4GiB headroom for the forward pass + full hidden-state capture. On a 48GB L40S
# every model used here (≤8B, ~15GB bf16) sits entirely on-GPU — no CPU/disk offload;
# on a 24GB card it still fits, just tighter. Portable across GPUs and model sizes.
_gpu_gib = int(torch.cuda.get_device_properties(0).total_memory / 1024**3)
_gpu_cap = max(_gpu_gib - 4, 8)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: f"{_gpu_cap}GiB", "cpu": "120GiB"},
    offload_folder="outputs/offload",  # last-resort fallback only if a model truly won't fit
    token=HF_READ_TOKEN,
)
model.eval()

# Support different config schemas (e.g., Gemma family and others).
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")
# Sanity check: with the cap above nothing should offload — every layer maps to GPU 0.
print(f"GPU cap: {_gpu_cap}GiB   device_map placements: {set(model.hf_device_map.values())}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-4B
Layers: 36, hidden_dim: 2560
GPU cap: 40GiB   device_map placements: {0}


## Part 2: Load Existing Judged Data

No knowledge test, generation, or judging is run — the supplement configs are re-labeled
from the existing `truthfulQA_full.csv` / `mmlu_full.csv`, which already hold every
`(config, votes_correct)` bucket.

In [3]:
tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
mmlu_full = pd.read_csv(MMLU_FULL_PATH)
print(f"tqa_full : {tqa_full.shape}")
print(f"mmlu_full: {mmlu_full.shape}")

# Sanity check: rows each supplement rule will select (TruthfulQA + MMLU)
for config, vc, label in SUPPLEMENT_LABEL_RULES:
    n = int(
        ((tqa_full["config"] == config)  & (tqa_full["votes_correct"] == vc)).sum()
        + ((mmlu_full["config"] == config) & (mmlu_full["votes_correct"] == vc)).sum()
    )
    print(f"  {label:20s} (config {config} / votes_correct {vc}): {n} rows")

tqa_full : (1139, 8)
mmlu_full: (19336, 8)
  natural_deception    (config A / votes_correct 0): 2499 rows
  capable_failed       (config B / votes_correct 6): 3340 rows
  deception_rejection  (config C / votes_correct 6): 1456 rows


## Part 3: Build Supplement Probe Dataset

Factual rows only — the new classes have no social scenario pairs, so
`include_social=False`. Labels come from `SUPPLEMENT_LABEL_RULES`.

In [4]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, None,
    SUPPLEMENT_PROBE_DATASET_PATH,
    rules=SUPPLEMENT_LABEL_RULES,
    include_social=False,
)

# Split thinking/answer: no-op for qwen2.5; strips <think>/gemma blocks for qwen3/gemma.
probe_dataset_split = split_thinking_responses(
    probe_dataset,
    save_path=SUPPLEMENT_PROBE_DATASET_SPLIT_PATH,
)

Saved probe_dataset (7295 rows) → probe_dataset_additional_config.csv

Label distribution:
label
capable_failed         3340
natural_deception      2499
deception_rejection    1456

Domain distribution:
domain
factual    7295
Rows with thinking blocks : 1 / 7295
Rows without thinking     : 7294 / 7295
Saved → probe_dataset_additional_config_split.csv


## Part 4: Extract Activations (new classes)

Last-token hidden state per layer, for the supplement rows only. Distinct filenames
(`activations_additional_config.npy`) sit alongside the 3-class files without overwriting them.
`run_extract_activations` derives its HuggingFace download name from the path, so the
3-class `activations.npy` is never pulled by mistake; on the first run nothing is on the
Hub and it extracts locally. Upload afterwards to reuse, or pass `hf_repo=""` to skip the Hub.

In [5]:
_df = probe_dataset_split.copy()
_df["response"] = _df["response_answer"]

# Drop degenerate ultra-long rows (e.g. a greedy-decoding repetition loop of tens of
# thousands of tokens) before extraction — they'd OOM the forward pass and are junk data.
# Legit rows here are ≤~1.5k tokens, so with MAX_EXTRACT_TOKENS=8192 this removes only true
# outliers. (extract_activations also left-truncates at max_length as a backstop.)
_df = drop_overlong_rows(_df, tokenizer, MAX_EXTRACT_TOKENS)

activations_arr, labels_arr = run_extract_activations(
    _df, model, tokenizer, DEVICE,
    SUPPLEMENT_ACTIVATIONS_PATH, SUPPLEMENT_LABELS_PATH, SUPPLEMENT_ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN, CHECKPOINT_EVERY,
    max_length=MAX_EXTRACT_TOKENS,
)
print(f"activations: {activations_arr.shape}")
print("Label counts:",
      {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items() if (labels_arr == v).any()})

[filter] dropped 2 row(s) > 8192 tokens (longest was 10048); kept 7293/7295
Local files not found. Downloading from maikurokosmos/llm-capability-based-deception-compliance ...
Download failed (EntryNotFoundError: 404 Client Error. (Request ID: Root=1-6a82be2d-6249b26f1a8b0d575e0fc3c4;e1f926c2-11b3-4b27-9807-fded26c3d5ac)

Entry Not Found for url: https://huggingface.co/datasets/maikurokosmos/llm-capability-based-deception-compliance/resolve/main/outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/activations_additional_config.npy.). Running extraction ...
Starting fresh: 7293 samples


Extracting activations:   0%|          | 0/7293 [00:00<?, ?it/s]

Extracted and saved: activations (7293, 36, 2560)
activations: (7293, 36, 2560)
Label counts: {'natural_deception': 2497, 'capable_failed': 3340, 'deception_rejection': 1456}


## Part 5: Combine with Base Activations + PCA

Probing needs all six classes: the three supplement classes just extracted **plus** the
original three (`truth` / `honest_mistake` / `deception`). The base activations load from
the original run (local, else HuggingFace) — no re-extraction if present. PCA is then fit
fresh on the union so components capture variance across all classes.

In [6]:
# Base 3-class activations from the original experiment (loaded, not re-extracted).
base_probe = split_thinking_responses(pd.read_csv(PROBE_DATASET_PATH))
base_probe["response"] = base_probe["response_answer"]
base_acts, base_labels = run_extract_activations(
    base_probe, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN, CHECKPOINT_EVERY,
    max_length=MAX_EXTRACT_TOKENS,
)

# Combine base (labels 0,1,2) + supplement (3,4,5); decode int labels -> strings
all_acts       = np.concatenate([base_acts, activations_arr], axis=0)
all_labels_int = np.concatenate([base_labels, labels_arr], axis=0)
_inv = {v: k for k, v in LABEL_MAP.items()}
all_labels = np.array([_inv[i] for i in all_labels_int])
print(f"Combined activations: {all_acts.shape}")
print("Class counts:", dict(zip(*[x.tolist() for x in np.unique(all_labels, return_counts=True)])))

# PCA fit on the union. hf_repo='' -> always compute locally (never pull the 3-class PCA).
acts_reduced = run_pca_reduction(
    all_acts, PCA_K,
    SUPPLEMENT_ACTIVATIONS_PCA_PATH, SUPPLEMENT_PCA_COMPONENTS_PATH, SUPPLEMENT_PCA_VARIANCE_PATH,
    hf_repo="", hf_token=HF_READ_TOKEN,
)

Rows with thinking blocks : 0 / 11416
Rows without thinking     : 11416 / 11416
Local files not found. Downloading from maikurokosmos/llm-capability-based-deception-compliance ...


outputs/qwen3-4b/non_thinking_mode/origi(…):   0%|          | 0.00/4.21G [00:00<?, ?B/s]

outputs/qwen3-4b/non_thinking_mode/origi(…):   0%|          | 0.00/91.5k [00:00<?, ?B/s]

Loading activations:   0%|          | 0/4208394368 [00:00<?, ?it/s]

Loading labels     :   0%|          | 0/91456 [00:00<?, ?it/s]

[HF] activations (11416, 36, 2560), labels (11416,)
Combined activations: (18709, 36, 2560)
Class counts: {'capable_failed': 3340, 'deception': 3496, 'deception_rejection': 1456, 'honest_mistake': 5518, 'natural_deception': 2497, 'truth': 2402}
Running PCA (64 components) across 36 layers ...
Saved activations_additional_config_pca64.npy       (18709, 36, 64)
Saved pca64_components_additional_config.npy (36, 64, 2560)
Saved pca64_explained_variance_additional_config.csv
Explained variance — mean: 0.706, min: 0.625, max: 0.888


## Part 6: Probing A — Multi-class Landscape (LR)

Which categories are easy/hard to tell apart. Two runs of the same `probe_all_layers`:
1. **Neutral 2×2** (`SUPPLEMENT_NEUTRAL_2X2`) → a clean 4×4 confusion matrix over
   *knowledge × answer-correctness*, no deceptive-prompt confound.
2. **All classes** → the full landscape (adds the deceptive-prompt classes).

In [7]:
# (i) neutral 2x2 — knowledge x answer-correctness
_mask = np.isin(all_labels, SUPPLEMENT_NEUTRAL_2X2)
res_2x2 = probe_all_layers(
    acts_reduced[_mask], all_labels[_mask],
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=SUPPLEMENT_MULTICLASS_LR_DIR / "probe_results_neutral_2x2.csv",
    checkpoint_path=SUPPLEMENT_MULTICLASS_LR_DIR / "checkpoint_neutral_2x2.pkl",
)

# (ii) all classes — full landscape
res_all = probe_all_layers(
    acts_reduced, all_labels,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=SUPPLEMENT_MULTICLASS_LR_DIR / "probe_results_all_classes.csv",
    checkpoint_path=SUPPLEMENT_MULTICLASS_LR_DIR / "checkpoint_all_classes.pkl",
)

for _name, _r in [("neutral 2x2", res_2x2), ("all classes", res_all)]:
    _best = _r.loc[_r["f1_macro"].idxmax()]
    print(f"{_name:12s}: best macro-F1={_best['f1_macro']:.3f} @ layer {int(_best['layer'])}")

3-way LR probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_neutral_2x2.csv (36 rows)


3-way LR probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_all_classes.csv (36 rows)
neutral 2x2 : best macro-F1=0.378 @ layer 22
all classes : best macro-F1=0.484 @ layer 21


## Part 7: Probing B — Targeted Binaries (LR, AUROC)

One sharp question per pair from `SUPPLEMENT_BINARY_PAIRS`. Each is just a two-class
subset fed to the generic binary probe (`drop_labels=()` because the subset is already the
exact pair). **AUROC ≈ 0.5 ⇒ the two classes are internally indistinguishable;** high AUROC
⇒ a separable internal signature.

In [8]:
binary_summary = []
for a, b, pos in SUPPLEMENT_BINARY_PAIRS:
    _mask = np.isin(all_labels, [a, b])
    _name = f"{a}__vs__{b}"
    _res = probe_all_layers_binary(
        acts_reduced[_mask], all_labels[_mask],
        pos_label=pos, drop_labels=(),
        n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
        output_path=SUPPLEMENT_BINARY_LR_DIR / f"{_name}.csv",
        checkpoint_path=SUPPLEMENT_BINARY_LR_DIR / f"checkpoint_{_name}.pkl",
    )
    _best = _res.loc[_res["auroc"].idxmax()]
    binary_summary.append({
        "pair": _name, "best_auroc": round(float(_best["auroc"]), 3),
        "best_layer": int(_best["layer"]), "n": int(_mask.sum()),
    })
    print(f"{_name:42s} best AUROC={_best['auroc']:.3f} @ layer {int(_best['layer'])}  (n={int(_mask.sum())})")

pd.DataFrame(binary_summary).to_csv(SUPPLEMENT_BINARY_LR_DIR / "summary_auroc.csv", index=False)
print(f"\nSaved summary -> {SUPPLEMENT_BINARY_LR_DIR / 'summary_auroc.csv'}")

binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved natural_deception__vs__deception.csv (36 rows)
natural_deception__vs__deception           best AUROC=1.000 @ layer 22  (n=5993)


binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved natural_deception__vs__honest_mistake.csv (36 rows)
natural_deception__vs__honest_mistake      best AUROC=0.559 @ layer 20  (n=8015)


binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved capable_failed__vs__honest_mistake.csv (36 rows)
capable_failed__vs__honest_mistake         best AUROC=0.809 @ layer 22  (n=8858)


binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved truth__vs__natural_deception.csv (36 rows)
truth__vs__natural_deception               best AUROC=0.840 @ layer 22  (n=4899)


binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved truth__vs__capable_failed.csv (36 rows)
truth__vs__capable_failed                  best AUROC=0.648 @ layer 15  (n=5742)

Saved summary -> outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/additional_config_binary_lr/summary_auroc.csv


## Part 8 (optional): Upload New Activations to HuggingFace

Pushes only the `*_additional_config*` `.npy` files into the **same repo tree** as the base
activations — base files are left untouched. Requires `HF_WRITE_TOKEN` in settings. Skip this
cell if you only need the results locally.

In [11]:
from huggingface_hub import HfApi

_patterns = ["**/*additional_config*.npy"]
_to_upload = [p.as_posix() for pat in _patterns for p in sorted(Path("outputs").glob(pat))]
print("To upload:", _to_upload)

if HF_WRITE_TOKEN and _to_upload:
    HfApi().upload_folder(
        folder_path="outputs",
        path_in_repo="outputs",
        repo_id=HF_ACTIVATIONS_REPO,
        repo_type="dataset",
        token=HF_WRITE_TOKEN,
        allow_patterns=_patterns,
    )
    print("Upload complete.")
else:
    print("Skipped (set HF_WRITE_TOKEN to enable, or nothing to upload).")

To upload: ['outputs/qwen2.5-7b-instruct/original_deception_prompt/activations_additional_config.npy', 'outputs/qwen2.5-7b-instruct/original_deception_prompt/activations_additional_config_pca64.npy', 'outputs/qwen2.5-7b-instruct/original_deception_prompt/labels_additional_config.npy', 'outputs/qwen2.5-7b-instruct/original_deception_prompt/pca64_components_additional_config.npy', 'outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/activations_additional_config.npy', 'outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/activations_additional_config_pca64.npy', 'outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/labels_additional_config.npy', 'outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/pca64_components_additional_config.npy']


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete.


## Part 9: Figures (paper)

Vector PDFs, each saved separately (no composites) under `additional_config_figures/`.
Per-layer line plots use **relative layer depth** on the x-axis so they stay comparable
across models with different layer counts (qwen2.5 → qwen3 → gemma-4).

1. **Binary AUROC by depth** — all `SUPPLEMENT_BINARY_PAIRS` in one panel + legend; dashed
   line at 0.5 = chance (AUROC ≈ 0.5 ⇒ internally indistinguishable).
2. **Multiclass macro-F1 by depth** — neutral 2×2 (4-class) and all classes (6-class); shows
   where separation peaks.
3. **Confusion matrix at the best layer** — the max-macro-F1 layer for each multiclass probe
   (neutral 2×2 and all classes), row-normalized (recall). Title notes layer, total layers,
   relative depth, and macro-F1.

Class order and figure paths live in `utils/settings.py` (`SUPPLEMENT_CM_ORDER_*`,
`SUPPLEMENT_FIG_*`).

In [10]:
from utils.plotting import plot_auroc, plot_macro_f1, plot_confusion_at_best_layer

SUPPLEMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load result CSVs
_binary = [(pd.read_csv(SUPPLEMENT_BINARY_LR_DIR / f"{a}__vs__{b}.csv"), f"{a} vs {b}")
           for a, b, _pos in SUPPLEMENT_BINARY_PAIRS]
_neutral = pd.read_csv(SUPPLEMENT_MULTICLASS_LR_DIR / "probe_results_neutral_2x2.csv")
_allcls  = pd.read_csv(SUPPLEMENT_MULTICLASS_LR_DIR / "probe_results_all_classes.csv")

# Fig 1 — binary AUROC vs relative layer depth (one panel, all pairs + legend)
plot_auroc(
    _binary, SUPPLEMENT_FIG_BINARY_AUROC,
    title="Binary probe AUROC by layer depth", x_axis="depth",
)

# Fig 2 — multiclass macro-F1 vs relative layer depth (neutral 2×2 + all classes)
plot_macro_f1(
    [(_neutral, "neutral 2×2 (4-class)"), (_allcls, "all classes (6-class)")],
    SUPPLEMENT_FIG_MACRO_F1,
    title="Multiclass probe macro-F1 by layer depth", x_axis="depth",
)

# Fig 3 — confusion matrix at the best (max macro-F1) layer, separate files
plot_confusion_at_best_layer(
    _neutral, SUPPLEMENT_FIG_CM_NEUTRAL_2X2,
    class_order=SUPPLEMENT_CM_ORDER_2X2, title_prefix="Neutral 2×2 — ",
)
plot_confusion_at_best_layer(
    _allcls, SUPPLEMENT_FIG_CM_ALL_CLASSES,
    class_order=SUPPLEMENT_CM_ORDER_ALL, title_prefix="All classes — ",
)

print("Figures written to", SUPPLEMENT_FIGURES_DIR)

Saved binary_auroc_by_depth.pdf
Saved multiclass_macro_f1_by_depth.pdf
Saved confusion_neutral_2x2_best_layer.pdf (best layer 22/36, depth 0.64)
Saved confusion_all_classes_best_layer.pdf (best layer 21/36, depth 0.61)
Figures written to outputs/qwen3-4b/non_thinking_mode/original_deception_prompt/additional_config_figures


## Notes / next steps

- **MLP** — deferred. Add later by calling `probe_all_layers_mlp(...)` (same signature +
  `hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES`) in Cell A, and a 2-class
  `probe_all_layers_mlp` per subset for the Cell B pairs.
- **Plots** — `utils/plotting.py` (`plot_macro_f1`, `plot_top_confusion_matrices`,
  `plot_auroc`) runs directly on these result CSVs when you want figures.
- **Cascaded / direct-3way / original binary** — intentionally not rerun; Cell B's
  targeted binaries replace the cascaded design.
- **Separate repo** — every new artifact is namespaced (`*_additional_config*` files and the
  `additional_config_multiclass_lr/` + `additional_config_binary_lr/` folders), kept apart
  from the base 3-class files.
- **Edit parameters in `utils/settings.py`** (`MODEL_ID`, `RUN_SLUG`,
  `SUPPLEMENT_BINARY_PAIRS`, `PCA_K`, ...), never in this notebook.